# Multi-Asset CTA Strategy V2 — Transition Strategy

## 07 — Transition State Machine and Portfolio Construction

Books 01–06 established the research chain:

$$
\text{Data}\rightarrow\text{Transition Events}\rightarrow\text{Features}\rightarrow\text{Frozen Probability}\rightarrow\text{Falsification}\rightarrow\text{Economic Validation}.
$$

Book 07 asks:

$$
\boxed{\text{Does the transition signal improve an implemented conventional CTA?}}
$$

This regenerated version corrects the initial Book 07 portfolio-construction flaw. The conventional CTA and transition sleeve are now **independent risk books**, each normalized and volatility-scaled separately **before** combination.

The hybrid is implemented in portfolio space as:

$$
R^{H}_{t}=(1-w_T)R^{C}_{t}+w_TR^{T}_{t},
$$

followed by one final portfolio-level volatility target. This prevents a sparse transition sleeve from being suppressed inside the conventional book's gross-normalization denominator.

### Primary comparison

1. **Conventional CTA**
2. **Transition Sleeve**
3. **Hybrid CTA + Transition Sleeve**

### State machine

$$
\text{ESTABLISHED}\rightarrow\text{CANDIDATE}\rightarrow\text{ACCUMULATING}\rightarrow
\begin{cases}\text{INVALIDATED}\\\text{CONFIRMED}\end{cases}
\rightarrow\text{CONVENTIONAL HAND-OFF}.
$$

Book 06 constraints are carried forward: no immediate maximum exposure; economic separation is clearer from roughly 21 observations onward; bear→bull and bull→bear may require different policies; and false positives make invalidation essential.

No Book 04 model retraining or feature discovery occurs here.


In [ ]:
# =========================================================
# 1) INSTALLS, IMPORTS, DRIVE, PATHS, CONFIG
# =========================================================
!pip -q install pyarrow
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
warnings.filterwarnings('ignore')

PROJECT = Path('/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2')
V201,V203,V204,V206,V207=[PROJECT/x for x in ['v2.01','v2.03','v2.04','v2.06','v2.07']]
CONFIG_DIR,DATA_DIR,RESULTS_DIR=V207/'config',V207/'data',V207/'results'
for p in [CONFIG_DIR,DATA_DIR,RESULTS_DIR]: p.mkdir(parents=True,exist_ok=True)
PATHS={
 'b4':V204/'data'/'v2_04_unweighted_predictions.parquet',
 'b3':V203/'data'/'v2_03_candidate_features.parquet',
 'b6':V206/'data'/'v2_06_event_economic_panel.parquet',
 'signal':V201/'data'/'processed'/'v2_01_signal_prices.parquet',
 'pnl':V201/'data'/'processed'/'v2_01_pnl_prices_prototype.parquet'}
CONFIG={
 'book':'V2.07','frozen_model':'RF_unweighted_C_plus_SC_no_MACD',
 'start':'2008-01-01','end':'2025-12-31','decision_frequency':'W-FRI',
 'slow_tsmom':252,'ma_fast':100,'ma_slow':300,'breakout':252,'votes':2,
 'asset_vol_lookback':63,'asset_target_vol':0.10,'asset_scale_cap':3.0,
 'sleeve_target_vol':0.10,'sleeve_vol_weeks':26,'sleeve_leverage_cap':2.0,
 'hybrid_target_vol':0.10,'hybrid_vol_weeks':26,'hybrid_leverage_cap':2.0,
 'transition_sleeve_weight':0.25,
 'policies':{
   'DELAYED_21_TOPQ':dict(min_age=21,max_age=126,qmin=.80,maxsig=.50,ramp=42,longmult=1.,shortmult=.50),
   'PROGRESSIVE_10_TOPQ':dict(min_age=10,max_age=126,qmin=.80,maxsig=.50,ramp=63,longmult=1.,shortmult=.50),
   'ASYMMETRIC_21':dict(min_age=21,max_age=126,qmin=.80,maxsig=.50,ramp=42,longmult=1.,shortmult=0.)},
 'cost_bps':[0,5,10,25]}
print(json.dumps(CONFIG,indent=2))


Mounted at /content/drive
{
  "book": "V2.07",
  "frozen_model": "RF_unweighted_C_plus_SC_no_MACD",
  "start": "2008-01-01",
  "end": "2025-12-31",
  "decision_frequency": "W-FRI",
  "slow_tsmom": 252,
  "ma_fast": 100,
  "ma_slow": 300,
  "breakout": 252,
  "votes": 2,
  "asset_vol_lookback": 63,
  "asset_target_vol": 0.1,
  "asset_scale_cap": 3.0,
  "sleeve_target_vol": 0.1,
  "sleeve_vol_weeks": 26,
  "sleeve_leverage_cap": 2.0,
  "hybrid_target_vol": 0.1,
  "hybrid_vol_weeks": 26,
  "hybrid_leverage_cap": 2.0,
  "transition_sleeve_weight": 0.25,
  "policies": {
    "DELAYED_21_TOPQ": {
      "min_age": 21,
      "max_age": 126,
      "qmin": 0.8,
      "maxsig": 0.5,
      "ramp": 42,
      "longmult": 1.0,
      "shortmult": 0.5
    },
    "PROGRESSIVE_10_TOPQ": {
      "min_age": 10,
      "max_age": 126,
      "qmin": 0.8,
      "maxsig": 0.5,
      "ramp": 63,
      "longmult": 1.0,
      "shortmult": 0.5
    },
    "ASYMMETRIC_21": {
      "min_age": 21,
      "max_age": 126,


## Architecture correction

The original Book 07 normalized the combined raw hybrid vector back to unit gross exposure, largely erasing the sparse transition sleeve. This version instead forms the conventional and transition sleeves independently, applies sleeve-level volatility targeting, combines the resulting return streams at a fixed 75/25 capital-risk budget, and then applies one final hybrid volatility target.

The 25% transition allocation therefore has economic meaning whenever the transition sleeve is active.


In [ ]:
# 2) LOAD / NORMALISE PRICES
def panel(df):
    x=df.copy(); lo={str(c).lower():c for c in x.columns}
    dc=next((lo[k] for k in ['date','datetime','timestamp'] if k in lo),None)
    mc=next((lo[k] for k in ['market','asset','name'] if k in lo),None)
    pc=next((lo[k] for k in ['price','close','signal_price','pnl_price','value'] if k in lo),None)
    if dc is not None and mc is not None and pc is not None:
        x[dc]=pd.to_datetime(x[dc]); return x.pivot_table(index=dc,columns=mc,values=pc,aggfunc='last').sort_index()
    if dc is not None: x[dc]=pd.to_datetime(x[dc]); x=x.set_index(dc)
    if not isinstance(x.index,pd.DatetimeIndex): x.index=pd.to_datetime(x.index)
    return x.apply(pd.to_numeric,errors='coerce').sort_index()
signal=panel(pd.read_parquet(PATHS['signal'])).loc[:CONFIG['end']]
pnl=panel(pd.read_parquet(PATHS['pnl'])).loc[:CONFIG['end']]
print('Signal',signal.shape,'P&L',pnl.shape)


Signal (10571, 53) P&L (10571, 52)


In [ ]:
# 3) LOAD FROZEN CANDIDATES / PROBABILITIES
if PATHS['b6'].exists():
    cand=pd.read_parquet(PATHS['b6']).copy()
else:
    cand=pd.read_parquet(PATHS['b3']).copy(); pred=pd.read_parquet(PATHS['b4']).copy()
    pred['candidate_date']=pd.to_datetime(pred['candidate_date'])
    pred=pred[pred['model'].astype(str)==CONFIG['frozen_model']]
    pc=next(c for c in ['prob_genuine','predicted_probability','probability','prediction'] if c in pred.columns)
    pred=pred.rename(columns={pc:'prob_genuine'})
    cand['candidate_date']=pd.to_datetime(cand['candidate_date'])
    keys=['candidate_date','market']+[c for c in ['candidate_direction','category'] if c in cand.columns and c in pred.columns]
    cand=cand.merge(pred[keys+['prob_genuine']].drop_duplicates(keys),on=keys,how='inner')
cand['candidate_date']=pd.to_datetime(cand['candidate_date']); cand['test_year']=cand.candidate_date.dt.year

def dsign(x):
    s=str(x).upper()
    if 'BEAR' in s and 'BULL' in s and s.index('BEAR')<s.index('BULL'): return 1.
    if 'BULL' in s and 'BEAR' in s and s.index('BULL')<s.index('BEAR'): return -1.
    try:return float(np.sign(float(x)))
    except:return np.nan
cand['transition_direction']=cand.candidate_direction.map(dsign)
cand=cand[(cand.candidate_date>=CONFIG['start'])&(cand.candidate_date<=CONFIG['end'])].copy()
# Architecture-only normalisation; Book08 replaces with fully causal threshold.
cand['prob_rank']=cand.groupby('test_year')['prob_genuine'].rank(method='average',pct=True)
print('Candidates',len(cand),'Markets',cand.market.nunique())


Candidates 2687 Markets 53


### Causality limitation

The state machine is causal with respect to market prices and candidate dates, but the within-year probability percentile uses the completed test year's candidate distribution. Book 07 therefore tests **architecture**, not a final deployable threshold. Book 08 must replace this with expanding or fixed historical thresholds.


In [ ]:
# 4) MARKET MATCHING + CONVENTIONAL CTA
def canon(s):
    s=str(s).lower()
    for a,b in [('&','and'),('/',''),('-',''),('_',''),(' ',''),('.',''),('^','')]: s=s.replace(a,b)
    return s
def match(m,cols):
    if m in cols:return m
    lu={canon(c):c for c in cols}; cm=canon(m)
    if cm in lu:return lu[cm]
    h=[c for c in cols if cm in canon(c) or canon(c) in cm]
    return h[0] if len(h)==1 else None
mmap={m:{'sig':match(m,signal.columns),'pnl':match(m,pnl.columns)} for m in sorted(cand.market.astype(str).unique())}

def convsig(s):
    s=s.dropna().astype(float)
    t=np.sign(s/s.shift(CONFIG['slow_tsmom'])-1)
    ma=np.sign(s.rolling(CONFIG['ma_fast']).mean()-s.rolling(CONFIG['ma_slow']).mean())
    hh=s.rolling(CONFIG['breakout']).max(); ll=s.rolling(CONFIG['breakout']).min(); br=np.sign(s-(hh+ll)/2)
    v=pd.concat([t,ma,br],axis=1); out=pd.Series(0.,index=s.index)
    out[(v>0).sum(axis=1)>=CONFIG['votes']]=1.; out[(v<0).sum(axis=1)>=CONFIG['votes']]=-1.
    out[v.isna().any(axis=1)]=np.nan
    return pd.DataFrame({'price':s,'conventional':out})
conv={m:convsig(signal[x['sig']]) for m,x in mmap.items() if x['sig'] is not None}
print('Conventional markets',len(conv))


Conventional markets 53


In [ ]:
# 5) TRANSITION STATE MACHINES
def make_state(m,pol):
    cv=conv[m]; dates=cv.index; cc=cand[cand.market.astype(str)==str(m)].sort_values('candidate_date')
    state=pd.Series('ESTABLISHED',index=dates,dtype=object); ts=pd.Series(0.,index=dates)
    direction=pd.Series(np.nan,index=dates); prob=pd.Series(np.nan,index=dates); rank=pd.Series(np.nan,index=dates); age_s=pd.Series(np.nan,index=dates)
    amap={}
    for _,r in cc.iterrows():
        i=int(dates.searchsorted(r.candidate_date,'left'))
        if i<len(dates): amap.setdefault(i,[]).append(r)
    active=None; start=None
    for i,dt in enumerate(dates):
        if i in amap: active=sorted(amap[i],key=lambda r:r.candidate_date)[-1]; start=i
        if active is None: continue
        d=float(active.transition_direction); q=float(active.prob_rank); p=float(active.prob_genuine); age=i-start; cs=cv.conventional.iloc[i]
        direction.iloc[i]=d; rank.iloc[i]=q; prob.iloc[i]=p; age_s.iloc[i]=age
        if np.isfinite(cs) and cs==d:
            state.iloc[i]='CONFIRMED'; active=None; start=None; continue
        if age>pol['max_age']:
            state.iloc[i]='INVALIDATED'; active=None; start=None; continue
        if age<pol['min_age'] or q<pol['qmin']:
            state.iloc[i]='CANDIDATE'; continue
        progress=min(1.,max(0.,(age-pol['min_age']+1)/max(1,pol['ramp'])))
        mult=pol['longmult'] if d>0 else pol['shortmult']
        ts.iloc[i]=d*pol['maxsig']*mult*progress; state.iloc[i]='ACCUMULATING'
    return pd.DataFrame({'state':state,'transition':ts,'direction':direction,'prob':prob,'rank':rank,'age':age_s})
states={pn:{m:make_state(m,p) for m in conv} for pn,p in CONFIG['policies'].items()}
print('State machines',sum(len(v) for v in states.values()))


State machines 159


In [ ]:
# 6) WEEKLY MARKET PANELS
weekly={}
for m,cv in conv.items():
    col=mmap[m]['pnl'] or mmap[m]['sig']; src='PNL_PROTOTYPE' if mmap[m]['pnl'] else 'SIGNAL_FALLBACK'
    ps=(pnl[col] if mmap[m]['pnl'] else signal[col]).dropna().astype(float)
    idx=cv.index.union(ps.index).sort_values(); x=pd.DataFrame(index=idx)
    x['p']=ps.reindex(idx).ffill(); x['r']=x.p.pct_change(); x['conv']=cv.conventional.reindex(idx).ffill()
    rv=x.r.rolling(CONFIG['asset_vol_lookback']).std()*np.sqrt(252)
    x['scale']=(CONFIG['asset_target_vol']/rv).clip(upper=CONFIG['asset_scale_cap'])
    wp=x.p.resample(CONFIG['decision_frequency']).last(); w=pd.DataFrame(index=wp.index)
    w['ret']=wp.pct_change(); w['conv']=x.conv.resample(CONFIG['decision_frequency']).last(); w['scale']=x.scale.resample(CONFIG['decision_frequency']).last(); w['source']=src
    weekly[m]=w
print('Weekly panels',len(weekly))


Weekly panels 53


## Independent sleeve construction

Market positions are lagged one weekly decision before earning returns. The conventional sleeve is normalized only against conventional positions. The transition sleeve is normalized only against transition positions and earns zero when inactive. Each sleeve then receives its own trailing volatility scaler before the hybrid is formed.


In [ ]:
# 7) RAW WEEKLY POSITION PANELS
panels={}
for pn in CONFIG['policies']:
    rr=[]
    for m,w in weekly.items():
        z=w.copy(); z['market']=m
        tr=states[pn][m].transition.resample(CONFIG['decision_frequency']).last().reindex(z.index).fillna(0.)
        st=states[pn][m].state.resample(CONFIG['decision_frequency']).last().reindex(z.index)
        di=states[pn][m].direction.resample(CONFIG['decision_frequency']).last().reindex(z.index)
        z['state']=st; z['direction']=di; z['conv_pos']=z.conv.fillna(0.)*z.scale; z['trans_pos']=tr*z.scale
        tmp=z.reset_index(); tmp=tmp.rename(columns={tmp.columns[0]:'date'}); rr.append(tmp)
    panels[pn]=pd.concat(rr,ignore_index=True)
print('Position panels complete')


Position panels complete


In [ ]:
# 8) INDEPENDENT SLEEVE ENGINE
def sleeve(panel,poscol):
    x=panel[(panel.date>=CONFIG['start'])&(panel.date<=CONFIG['end'])].sort_values(['market','date']).copy()
    x['lp']=x.groupby('market')[poscol].shift(1)
    den=x.groupby('date').lp.transform(lambda s:s.abs().sum())
    x['w']=np.where(den>0,x.lp/den,0.)
    x['contrib']=x.w*x.ret.fillna(0.)
    raw=x.groupby('date').contrib.sum().sort_index()
    ww=x.pivot(index='date',columns='market',values='w').fillna(0.).sort_index()
    turn=ww.diff().abs().sum(axis=1).fillna(0.)
    vol=raw.rolling(CONFIG['sleeve_vol_weeks']).std()*np.sqrt(52)
    lev=(CONFIG['sleeve_target_vol']/vol).clip(upper=CONFIG['sleeve_leverage_cap']).shift(1).fillna(1.)
    summary=pd.DataFrame({'raw_return':raw,'leverage':lev,'return':raw*lev,'turnover':turn*lev,'active_markets':(ww!=0).sum(axis=1)})
    return {'summary':summary,'weights':ww,'detail':x}
sleeves={}
for pn,x in panels.items():
    sleeves[(pn,'CONVENTIONAL')]=sleeve(x,'conv_pos'); sleeves[(pn,'TRANSITION')]=sleeve(x,'trans_pos')
print('Sleeves',len(sleeves))


Sleeves 6


In [ ]:
# 9) HYBRID IN PORTFOLIO SPACE
def hybrid(c,t):
    cr=c['summary']['return'].rename('conv'); tr=t['summary']['return'].rename('trans')
    z=pd.concat([cr,tr],axis=1).fillna(0.); wt=CONFIG['transition_sleeve_weight']; wc=1-wt
    z['pre']=wc*z.conv+wt*z.trans
    vol=z.pre.rolling(CONFIG['hybrid_vol_weeks']).std()*np.sqrt(52)
    z['lev']=(CONFIG['hybrid_target_vol']/vol).clip(upper=CONFIG['hybrid_leverage_cap']).shift(1).fillna(1.)
    z['return']=z.pre*z.lev
    tc=c['summary']['turnover'].reindex(z.index).fillna(0.); tt=t['summary']['turnover'].reindex(z.index).fillna(0.)
    z['turnover']=(wc*tc+wt*tt)*z.lev
    return z
hybrids={pn:hybrid(sleeves[(pn,'CONVENTIONAL')],sleeves[(pn,'TRANSITION')]) for pn in CONFIG['policies']}
print('Hybrids',len(hybrids))


Hybrids 3


In [ ]:
# 10) PERFORMANCE + INCREMENTAL VALUE
def mdd(r):
    w=(1+r.fillna(0)).cumprod(); return float((w/w.cummax()-1).min())
def metrics(r):
    r=r.dropna(); n=len(r)
    if not n:return {}
    ar=(1+r).prod()**(52/n)-1; av=r.std()*np.sqrt(52); dn=r[r<0].std()*np.sqrt(52)
    return {'weeks':n,'annual_return':ar,'annual_vol':av,'sharpe_0rf':ar/av if av>0 else np.nan,'sortino_0rf':ar/dn if dn>0 else np.nan,'max_drawdown':mdd(r),'positive_week_rate':(r>0).mean()}
perf=[]; inc=[]
for pn in CONFIG['policies']:
    c=sleeves[(pn,'CONVENTIONAL')]['summary']; t=sleeves[(pn,'TRANSITION')]['summary']; h=hybrids[pn]
    for sn,r,to in [('CONVENTIONAL',c['return'],c.turnover),('TRANSITION',t['return'],t.turnover),('HYBRID',h['return'],h.turnover)]: perf.append({'policy':pn,'strategy':sn,**metrics(r),'mean_weekly_turnover':to.mean()})
    z=pd.concat([c['return'].rename('c'),t['return'].rename('t'),h['return'].rename('h')],axis=1).dropna(); cm,tm,hm=metrics(z.c),metrics(z.t),metrics(z.h); d=z.h-z.c
    inc.append({'policy':pn,'weeks':len(z),'conventional_annual_return':cm['annual_return'],'transition_annual_return':tm['annual_return'],'hybrid_annual_return':hm['annual_return'],'delta_annual_return':hm['annual_return']-cm['annual_return'],'conventional_sharpe':cm['sharpe_0rf'],'transition_sharpe':tm['sharpe_0rf'],'hybrid_sharpe':hm['sharpe_0rf'],'delta_sharpe':hm['sharpe_0rf']-cm['sharpe_0rf'],'conventional_max_dd':cm['max_drawdown'],'transition_max_dd':tm['max_drawdown'],'hybrid_max_dd':hm['max_drawdown'],'delta_max_dd':hm['max_drawdown']-cm['max_drawdown'],'incremental_mean_weekly':d.mean(),'incremental_positive_week_rate':(d>0).mean(),'conv_transition_correlation':z.c.corr(z.t),'conv_hybrid_correlation':z.c.corr(z.h)})
performance=pd.DataFrame(perf); incremental=pd.DataFrame(inc)
display(performance); display(incremental)


,policy,strategy,weeks,annual_return,annual_vol,sharpe_0rf,sortino_0rf,max_drawdown,positive_week_rate,mean_weekly_turnover
0,DELAYED_21_TOPQ,CONVENTIONAL,939,0.025626,0.076855,0.333425,0.446523,-0.164016,0.547391,0.233427
1,DELAYED_21_TOPQ,TRANSITION,939,0.029752,0.158382,0.187849,0.220780,-0.360413,0.257721,0.171657
2,DELAYED_21_TOPQ,HYBRID,939,0.047857,0.104086,0.459780,0.648581,-0.168940,0.545261,0.367571
3,PROGRESSIVE_10_TOPQ,CONVENTIONAL,939,0.025626,0.076855,0.333425,0.446523,-0.164016,0.547391,0.233427
4,PROGRESSIVE_10_TOPQ,TRANSITION,939,-0.012040,0.156689,-0.076842,-0.065309,-0.413890,0.310969,0.218484
5,PROGRESSIVE_10_TOPQ,HYBRID,939,0.034593,0.102623,0.337087,0.451741,-0.234366,0.542066,0.391327
6,ASYMMETRIC_21,CONVENTIONAL,939,0.025626,0.076855,0.333425,0.446523,-0.164016,0.547391,0.233427
7,ASYMMETRIC_21,TRANSITION,939,0.003868,0.148404,0.026061,0.029734,-0.452656,0.206603,0.147641
8,ASYMMETRIC_21,HYBRID,939,0.038508,0.103841,0.370836,0.526177,-0.187190,0.546326,0.359601


,policy,weeks,conventional_annual_return,transition_annual_return,hybrid_annual_return,delta_annual_return,conventional_sharpe,transition_sharpe,hybrid_sharpe,delta_sharpe,conventional_max_dd,transition_max_dd,hybrid_max_dd,delta_max_dd,incremental_mean_weekly,incremental_positive_week_rate,conv_transition_correlation,conv_hybrid_correlation
0,DELAYED_21_TOPQ,939,0.025626,0.029752,0.047857,0.022231,0.333425,0.187849,0.459780,0.126355,-0.164016,-0.360413,-0.168940,-0.004925,0.000460,0.542066,-0.182702,0.771881
1,PROGRESSIVE_10_TOPQ,939,0.025626,-0.012040,0.034593,0.008967,0.333425,-0.076842,0.337087,0.003662,-0.164016,-0.413890,-0.234366,-0.070351,0.000212,0.538871,-0.218071,0.757656
2,ASYMMETRIC_21,939,0.025626,0.003868,0.038508,0.012882,0.333425,0.026061,0.370836,0.037410,-0.164016,-0.452656,-0.187190,-0.023174,0.000287,0.532481,-0.178597,0.787782


In [ ]:
# 11) COST SENSITIVITY
rows=[]
for pn in CONFIG['policies']:
    c=sleeves[(pn,'CONVENTIONAL')]['summary']; t=sleeves[(pn,'TRANSITION')]['summary']; h=hybrids[pn]
    streams={'CONVENTIONAL':(c['return'],c.turnover),'TRANSITION':(t['return'],t.turnover),'HYBRID':(h['return'],h.turnover)}
    for sn,(r,to) in streams.items():
        for b in CONFIG['cost_bps']:
            rows.append({'policy':pn,'strategy':sn,'cost_bps':b,**metrics(r-to*b/10000),'mean_weekly_turnover':to.mean()})
costs=pd.DataFrame(rows); display(costs)


,policy,strategy,cost_bps,weeks,annual_return,annual_vol,sharpe_0rf,sortino_0rf,max_drawdown,positive_week_rate,mean_weekly_turnover
0,DELAYED_21_TOPQ,CONVENTIONAL,0,939,0.025626,0.076855,0.333425,0.446523,-0.164016,0.547391,0.233427
1,DELAYED_21_TOPQ,CONVENTIONAL,5,939,0.019424,0.076828,0.252825,0.338574,-0.172565,0.539936,0.233427
2,DELAYED_21_TOPQ,CONVENTIONAL,10,939,0.013259,0.076807,0.172629,0.231189,-0.181028,0.537806,0.233427
3,DELAYED_21_TOPQ,CONVENTIONAL,25,939,-0.005020,0.076776,-0.065387,-0.087514,-0.280477,0.523962,0.233427
4,DELAYED_21_TOPQ,TRANSITION,0,939,0.029752,0.158382,0.187849,0.220780,-0.360413,0.257721,0.171657
5,DELAYED_21_TOPQ,TRANSITION,5,939,0.025206,0.158098,0.159433,0.193478,-0.363823,0.255591,0.171657
6,DELAYED_21_TOPQ,TRANSITION,10,939,0.020677,0.157829,0.131011,0.159146,-0.367216,0.253461,0.171657
7,DELAYED_21_TOPQ,TRANSITION,25,939,0.007194,0.157115,0.045785,0.055741,-0.377299,0.247071,0.171657
8,DELAYED_21_TOPQ,HYBRID,0,939,0.047857,0.104086,0.459780,0.648581,-0.168940,0.545261,0.367571
9,DELAYED_21_TOPQ,HYBRID,5,939,0.037907,0.103988,0.364530,0.512662,-0.175222,0.537806,0.367571


In [ ]:
# 12) YEARLY PERFORMANCE + INCREMENTAL ATTRIBUTION
yr=[]; yi=[]
for pn in CONFIG['policies']:
    c=sleeves[(pn,'CONVENTIONAL')]['summary']['return']; t=sleeves[(pn,'TRANSITION')]['summary']['return']; h=hybrids[pn]['return']
    for sn,r in [('CONVENTIONAL',c),('TRANSITION',t),('HYBRID',h)]:
        for y,g in r.dropna().groupby(r.index.year): yr.append({'policy':pn,'strategy':sn,'year':int(y),**metrics(g)})
    z=pd.concat([c.rename('c'),t.rename('t'),h.rename('h')],axis=1).dropna(); z['inc']=z.h-z.c
    for y,g in z.groupby(z.index.year): yi.append({'policy':pn,'year':int(y),'weeks':len(g),'conventional_return_sum':g.c.sum(),'transition_return_sum':g.t.sum(),'hybrid_return_sum':g.h.sum(),'incremental_return_sum':g.inc.sum(),'incremental_positive_week_rate':(g.inc>0).mean()})
yearly=pd.DataFrame(yr); yearly_incremental=pd.DataFrame(yi); display(yearly_incremental.head())


,policy,year,weeks,conventional_return_sum,transition_return_sum,hybrid_return_sum,incremental_return_sum,incremental_positive_week_rate
0,DELAYED_21_TOPQ,2008,52,0.157157,0.216982,0.213615,0.056458,0.423077
1,DELAYED_21_TOPQ,2009,52,-0.076232,0.077936,-0.035437,0.040795,0.519231
2,DELAYED_21_TOPQ,2010,53,0.059223,-0.171357,-0.013036,-0.072259,0.433962
3,DELAYED_21_TOPQ,2011,52,-0.066649,0.275127,0.032657,0.099306,0.596154
4,DELAYED_21_TOPQ,2012,52,-0.031652,0.016120,-0.037969,-0.006317,0.615385


In [ ]:
# 13) STATE / ACTIVE-SLEEVE / ATTRIBUTION DIAGNOSTICS
sd=[]; active=[]; attrib=[]
catmap=cand[['market','category']].drop_duplicates('market').set_index('market').category.to_dict()
for pn,mm in states.items():
    for m,s in mm.items():
        vc=s.state.value_counts(); sd.append({'policy':pn,'market':m,'category':catmap.get(m,'UNKNOWN'),'candidate_obs':int(vc.get('CANDIDATE',0)),'accumulating_obs':int(vc.get('ACCUMULATING',0)),'confirmed_obs':int(vc.get('CONFIRMED',0)),'invalidated_obs':int(vc.get('INVALIDATED',0)),'mean_abs_transition_signal':s.transition.abs().mean(),'max_abs_transition_signal':s.transition.abs().max()})
    ts=sleeves[(pn,'TRANSITION')]['summary']; a=ts.active_markets>0
    active.append({'policy':pn,'weeks_total':len(ts),'weeks_transition_active':int(a.sum()),'active_week_rate':a.mean(),'mean_active_markets_when_active':ts.loc[a,'active_markets'].mean() if a.any() else np.nan,'median_active_markets_when_active':ts.loc[a,'active_markets'].median() if a.any() else np.nan,'mean_abs_transition_return_when_active':ts.loc[a,'return'].abs().mean() if a.any() else np.nan,'mean_transition_turnover':ts.turnover.mean()})
    det=sleeves[(pn,'TRANSITION')]['detail'].copy(); det['category']=det.market.map(catmap); det['direction_name']=np.where(det.direction>0,'BEAR_TO_BULL',np.where(det.direction<0,'BULL_TO_BEAR','NONE')); det['transition_contribution']=det.w*det.ret.fillna(0.)
    for (cat,d),g in det.groupby(['category','direction_name'],dropna=False): attrib.append({'policy':pn,'category':cat,'direction_name':d,'observations':len(g),'active_observations':int((g.w!=0).sum()),'mean_weekly_contribution':g.transition_contribution.mean(),'sum_contribution':g.transition_contribution.sum()})
state_diag=pd.DataFrame(sd); active_diag=pd.DataFrame(active); attribution=pd.DataFrame(attrib)
display(active_diag)


,policy,weeks_total,weeks_transition_active,active_week_rate,mean_active_markets_when_active,median_active_markets_when_active,mean_abs_transition_return_when_active,mean_transition_turnover
0,DELAYED_21_TOPQ,939,487,0.518637,1.963039,2.0,0.016837,0.171657
1,PROGRESSIVE_10_TOPQ,939,599,0.637913,2.178631,2.0,0.015300,0.218484
2,ASYMMETRIC_21,939,411,0.437700,1.832117,1.0,0.018235,0.147641


## Completion Gate

Book 07 advances only if at least one **pre-specified** transition architecture adds credible value to the conventional CTA. The hybrid, not the standalone transition sleeve, is the key object.

We want higher hybrid Sharpe and/or better return/drawdown behaviour, contribution across multiple years, manageable turnover/cost sensitivity, sensible state activity, and results that are no longer mechanically identical to the conventional CTA.

Book 07 does **not** freeze a deployable strategy. Book 08 must replace the annual percentile normalisation with fully causal expanding/fixed historical thresholds and stress-test threshold choice, sleeve weight, delay, ramp speed, direction asymmetry, asset-class exclusions, Bitcoin exclusion, crises, concentration and transaction costs.

If the corrected hybrid still adds no material value, the transition sleeve should be rejected rather than rescued through specification mining.


In [ ]:
# 14) SAVE OUTPUTS
retrows=[]
for pn in CONFIG['policies']:
    c=sleeves[(pn,'CONVENTIONAL')]['summary']; t=sleeves[(pn,'TRANSITION')]['summary']; h=hybrids[pn]
    q=pd.DataFrame(index=h.index); q['conventional_return']=c['return'].reindex(q.index); q['transition_return']=t['return'].reindex(q.index); q['hybrid_pre_target_return']=h.pre; q['hybrid_return']=h['return']; q['conventional_turnover']=c.turnover.reindex(q.index); q['transition_turnover']=t.turnover.reindex(q.index); q['hybrid_turnover']=h.turnover; q['transition_active_markets']=t.active_markets.reindex(q.index); q['policy']=pn
    z=q.reset_index(); z=z.rename(columns={z.columns[0]:'date'}); retrows.append(z)
portfolio_returns=pd.concat(retrows,ignore_index=True); market_map=pd.DataFrame([{'market':m,**x} for m,x in mmap.items()])
outputs={
 'performance':RESULTS_DIR/'v2_07_performance_summary.csv','incremental':RESULTS_DIR/'v2_07_hybrid_incremental_value.csv','costs':RESULTS_DIR/'v2_07_cost_sensitivity.csv','yearly':RESULTS_DIR/'v2_07_yearly_performance.csv','yearly_incremental':RESULTS_DIR/'v2_07_yearly_incremental_attribution.csv','states':RESULTS_DIR/'v2_07_state_diagnostics.csv','active':RESULTS_DIR/'v2_07_active_sleeve_diagnostics.csv','attribution':RESULTS_DIR/'v2_07_transition_attribution.csv','returns':DATA_DIR/'v2_07_portfolio_returns.parquet','market_map':RESULTS_DIR/'v2_07_market_map.csv','config':CONFIG_DIR/'v2_07_config.json'}
performance.to_csv(outputs['performance'],index=False); incremental.to_csv(outputs['incremental'],index=False); costs.to_csv(outputs['costs'],index=False); yearly.to_csv(outputs['yearly'],index=False); yearly_incremental.to_csv(outputs['yearly_incremental'],index=False); state_diag.to_csv(outputs['states'],index=False); active_diag.to_csv(outputs['active'],index=False); attribution.to_csv(outputs['attribution'],index=False); portfolio_returns.to_parquet(outputs['returns'],index=False); market_map.to_csv(outputs['market_map'],index=False)
with open(outputs['config'],'w') as f: json.dump(CONFIG,f,indent=2)
for k,p in outputs.items(): print(k,p)


performance /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.07/results/v2_07_performance_summary.csv
incremental /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.07/results/v2_07_hybrid_incremental_value.csv
costs /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.07/results/v2_07_cost_sensitivity.csv
yearly /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.07/results/v2_07_yearly_performance.csv
yearly_incremental /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.07/results/v2_07_yearly_incremental_attribution.csv
states /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.07/results/v2_07_state_diagnostics.csv
active /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.07/results/v2_07_active_sleeve_diagnostics.csv
attribution /content/drive/MyDrive/Cola

## Outputs to upload for analysis

### Required
1. `v2_07_performance_summary.csv`
2. `v2_07_hybrid_incremental_value.csv`
3. `v2_07_cost_sensitivity.csv`
4. `v2_07_yearly_performance.csv`
5. `v2_07_yearly_incremental_attribution.csv`
6. `v2_07_state_diagnostics.csv`
7. `v2_07_active_sleeve_diagnostics.csv`

### Preferred
8. `v2_07_transition_attribution.csv`
9. `v2_07_portfolio_returns.parquet`
10. `v2_07_market_map.csv`

Write the Book 07 Research Outcome only after these corrected independent-sleeve results have been reviewed.


## Research Outcome

Book 07 tested whether the frozen Book 04 transition probability could improve an implemented conventional CTA once converted into an explicit transition state machine and an independently risk-scaled portfolio sleeve.

The regenerated implementation corrected the original portfolio-construction flaw by forming the conventional CTA and transition sleeve as **independent risk books**, normalizing and volatility-scaling each separately before combining them in portfolio space.

This produced a materially different and economically informative result.

### Conventional baseline

The conventional CTA produced approximately:

- **2.56% annual return**
- **7.69% annualized volatility**
- **Sharpe 0.333**
- **maximum drawdown −16.40%**

This is materially weaker than the earlier V1 CTA and should therefore be interpreted as the transparent Book 07 benchmark architecture rather than as the project's best possible trend-following implementation.

### Surviving transition architecture

The clear winner was **DELAYED_21_TOPQ**.

Its hybrid portfolio produced approximately:

- **4.79% annual return**
- **Sharpe 0.460**
- **maximum drawdown −16.89%**

Relative to the conventional CTA, this represents approximately:

- **+2.22 percentage points annual return**
- **+0.126 Sharpe**
- only a modest deterioration in maximum drawdown.

The transition sleeve also exhibited a **negative correlation of approximately −0.18 with the conventional CTA**, indicating genuine diversification rather than simple repackaging of the existing trend signal.

The hybrid/conventional relationship is therefore economically meaningful rather than mechanically identical.

### Falsification across transition policies

The alternative transition policies did not perform comparably.

**PROGRESSIVE_10_TOPQ** produced only a negligible Sharpe improvement relative to the conventional CTA and materially worsened drawdown. Its standalone transition sleeve lost approximately **−1.20% annually**.

This is consistent with Book 06: transition probability appears to identify a developing reversal process, but attempting to monetize it too early sacrifices much of the value.

**ASYMMETRIC_21** improved the conventional CTA modestly but remained clearly inferior to DELAYED_21_TOPQ. Restricting anticipatory exposure to bear→bull transitions therefore did not improve the portfolio enough to justify removing bull→bear participation entirely.

Accordingly, the preferred architecture is not simply “long-only transition alpha.” The evidence instead supports **delayed, probability-filtered transition exposure with asymmetric but non-zero short-side participation**.

### Temporal breadth

The DELAYED_21_TOPQ hybrid generated positive incremental return relative to the conventional CTA in approximately **11 of 18 OOS years**, with a median annual incremental contribution of roughly **+2.4%**.

Strong positive incremental years included:

- 2011
- 2016
- 2018
- 2020
- 2024

The transition sleeve also experienced materially negative years, including 2010 and 2023. The result is therefore not uniformly positive and should not be interpreted as a stable independent alpha stream in every regime.

### Portfolio sparsity and state behaviour

The corrected transition sleeve was active in approximately **52% of portfolio weeks**, with only around **2 active markets on average when active**.

This confirms that V2 is a **sparse transition sleeve**, not a continuously invested directional strategy. This sparsity explains why independent sleeve construction was essential: combining transition positions inside the conventional book's gross-normalization denominator materially diluted the intended transition risk budget.

### Asset-class attribution

Transition contributions are heterogeneous across asset classes and directions.

Equity-index transitions and some commodity bear→bull transitions contribute positively, while some bonds/rates and FX transition groups are weaker or negative.

These results are informative but are **not used to re-optimize the architecture in Book 07**. Asset-class exclusions are reserved for Book 08 robustness tests to avoid specification mining.

### Transaction-cost warning

Turnover remains a major implementation concern.

For DELAYED_21_TOPQ, hybrid weekly turnover is approximately **36.8%**. Under the schematic cost model:

- at **5 bps**, hybrid Sharpe falls from approximately **0.460 to 0.365**;
- at **10 bps**, hybrid Sharpe falls to approximately **0.270**;
- at **25 bps**, the strategy is effectively destroyed.

This is partly attributable to the conventional prototype itself, which also has high weekly turnover.

Book 08 must therefore treat turnover and implementation costs as a primary falsification dimension rather than as a secondary sensitivity check.

### Interpretation

Book 07 provides the first portfolio-level evidence that the research chain has crossed from prediction into incremental portfolio value:

\[
\text{Book 04 Transition Probability}
\rightarrow
\text{Book 06 Economic Information}
\rightarrow
\text{Book 07 Portfolio Improvement}.
\]

However, the surviving architecture is still **provisional**.

The principal remaining issue is that the Book 07 top-quintile probability filter uses the completed OOS year's probability distribution. This is not fully causal and cannot be accepted in a production specification.

Book 08 must therefore attempt to falsify DELAYED_21_TOPQ using fully causal probability thresholds, transaction costs, parameter perturbations, subperiods, asset-class exclusions, Bitcoin exclusion, concentration tests and alternative implementation assumptions.

### Conclusion

Book 07 supports a delayed transition overlay rather than immediate reversal trading.

The surviving architecture is:

\[
\boxed{
\text{DELAYED\_21\_TOPQ}
}
\]

with:

- approximately 21 local observations before anticipatory accumulation begins;
- progressive exposure accumulation thereafter;
- high transition-probability filtering;
- full bear→bull transition budget;
- reduced but non-zero bull→bear transition budget;
- independent transition-sleeve normalization and volatility targeting;
- portfolio-space combination with the conventional CTA;
- immediate hand-off at conventional confirmation.

**Status: FROZEN as the V2 portfolio-construction architecture. DELAYED_21_TOPQ PASSES the portfolio-construction gate. Earlier 10-observation accumulation is REJECTED; bear→bull-only implementation is NOT preferred. Selection remains provisional pending fully causal thresholding and final robustness testing in Book 08.**
